# 🏦 Banking AI-Agent: Full App on Google Colab

This notebook sets up the complete Banking AI-Agent system on Google Colab, including:
1. **Ollama Backend** (running `gpt-oss:20b`)
2. **FastAPI API Server** (Agent Orchestrator)
3. **Streamlit Web Frontend** (with Voice support)

It uses `localtunnel` to provide public URLs for both the API and the Web Interface.

### Step 1: Clone the Repository
Cloning the project and entering the exercise directory.

In [ ]:
!git clone https://github.com/TheWallOnFire/NLP-Project2.git
%cd NLP-Project2/Ex3

### Step 1.5: Apply Local Improvements
Since some of the latest improvements (Frontend, enhanced Orchestrator) are still in local development, we apply them here directly.

In [ ]:
import os
os.makedirs('frontend', exist_ok=True)

files_to_patch = {
    'frontend/app.py': r"""import streamlit as st
import requests
import json
import base64
from streamlit_mic_recorder import mic_recorder

# Set page config for a premium look
st.set_page_config(
    page_title="Banking AI-Agent",
    page_icon="🏦",
    layout="centered"
)

# Custom CSS for better aesthetics
st.markdown("""
    <style>
    .main {
        background-color: #f5f7f9;
    }
    .stButton>button {
        width: 100%;
        border-radius: 10px;
        height: 3em;
        background-color: #1e3a8a;
        color: white;
    }
    .stTextInput>div>div>input {
        border-radius: 10px;
    }
    .response-card {
        padding: 20px;
        background-color: white;
        border-radius: 15px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        margin-bottom: 20px;
    }
    .trace-section {
        background-color: #e5e7eb;
        padding: 15px;
        border-radius: 10px;
        margin-top: 10px;
    }
    .priority-high { color: #dc2626; font-weight: bold; }
    .priority-medium { color: #d97706; font-weight: bold; }
    .priority-low { color: #059669; font-weight: bold; }
    </style>
""", unsafe_allow_html=True)

def speak_text(text):
    js = f"""
    <script>
    var msg = new SpeechSynthesisUtterance("{text.replace('"', "'")}");
    window.speechSynthesis.speak(msg);
    </script>
    """
    st.components.v1.html(js, height=0)

def main():
    st.title("🏦 Banking Assistant AI")
    st.markdown("How can I help you with your account today?")

    if 'query_input' not in st.session_state:
        st.session_state.query_input = ""

    st.write("🎤 **Voice Input**")
    audio = mic_recorder(
        start_prompt="Click to Speak",
        stop_prompt="Stop Recording",
        just_once=True,
        use_container_width=True,
        key='recorder'
)

    if audio:
        st.info("Audio recorded successfully! (STT integration pending - please use text for now)")

    query = st.text_input("Enter your request:", value=st.session_state.query_input, placeholder="e.g., I lost my credit card...")

    if st.button("Process Request") and query:
        with st.spinner("Processing..."):
            try:
                response = requests.post("http://localhost:8000/process", json={"query": query})
                response.raise_for_status()
                data = response.json()
                st.markdown("### 🤖 Assistant's Response")
                st.markdown(f'<div class="response-card">{data["response"]}</div>', unsafe_allow_html=True)
                speak_text(data["response"])
                with st.expander("🔍 View Agentic Workflow Trace"):
                    st.json(data["trace"])
            except Exception as e:
                st.error(f"Error: {e}")

if __name__ == "__main__":
    main()""",
    'app/nodes/policy_node.py': r"""from ..core.schemas import PolicyResult
from ..data.policies import BANKING_POLICIES

class PolicyNode:
    def process(self, intent: str) -> PolicyResult:
        policy = BANKING_POLICIES.get(intent, BANKING_POLICIES.get("general"))
        if not policy:
            policy = {"snippet": "Please contact support.", "source": "General"}
        return PolicyResult(policy_snippet=policy["snippet"], source=policy["source"])""",
    'app/core/schemas.py': r"""from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class RoutingDecision(str, Enum):
    SEND = "send"
    ASK_MORE = "ask_more"
    ESCALATE = "escalate"

class QueryRequest(BaseModel):
    query: str
    session_id: Optional[str] = None

class IntentResult(BaseModel):
    intent: str
    confidence: float

class PriorityResult(BaseModel):
    level: Priority
    reason: str

class PolicyResult(BaseModel):
    policy_snippet: str
    source: str

class DraftResult(BaseModel):
    draft_response: str
    missing_information: List[str] = []
    next_action: Optional[str] = None

class ValidationResult(BaseModel):
    is_valid: bool
    feedback: Optional[str] = None

class RouterResult(BaseModel):
    decision: RoutingDecision
    explanation: str

class AgentTrace(BaseModel):
    intent: Optional[IntentResult] = None
    priority: Optional[PriorityResult] = None
    policy: Optional[PolicyResult] = None
    draft: Optional[DraftResult] = None
    validation: Optional[ValidationResult] = None
    router: Optional[RouterResult] = None

class QueryResponse(BaseModel):
    query: str
    response: str
    trace: AgentTrace"""
}

for path, content in files_to_patch.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)
    print(f"✅ Patched {path}")

### Step 2: Install and Start Ollama
We start Ollama in the background to serve the LLM.

In [ ]:
# Install dependencies for Ollama
!apt-get update -qq && apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import threading
import time

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"], env=env)

threading.Thread(target=run_ollama_serve, daemon=True).start()
time.sleep(5)  # Wait for server to start

### Step 3: Pull the Model
Downloading the specialized `gpt-oss:20b` model.

In [ ]:
!ollama pull gpt-oss:20b

### Step 4: Install Dependencies
Installing all required Python packages for both backend and frontend.

In [ ]:
!pip install -r requirements.txt
!pip install streamlit streamlit_mic_recorder

### Step 5: Expose Ports via Tunnels
We use `localtunnel` to create public URLs for the API (8000) and Frontend (8501).

**IMPORTANT:** When you click the links below, you will need to enter the **Endpoint IP** provided to bypass the security screen.

In [ ]:
!npm install -g localtunnel

import urllib
import subprocess
import time

endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("COPY THIS IP ADDRESS (Localtunnel Password):", endpoint_ip)

# Start API Tunnel (8000)
api_tunnel = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
time.sleep(2)
print("\nYOUR PUBLIC API URL:", api_tunnel.stdout.readline().decode('utf-8').strip())

# Start Frontend Tunnel (8501)
frontend_tunnel = subprocess.Popen(["lt", "--port", "8501"], stdout=subprocess.PIPE)
time.sleep(2)
print("YOUR PUBLIC FRONTEND URL:", frontend_tunnel.stdout.readline().decode('utf-8').strip())

### Step 6: Launch the Application
Starting the backend in the background and the Streamlit frontend in the foreground.

In [ ]:
import subprocess

print("Starting FastAPI Backend...")
subprocess.Popen(["python", "run.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starting Streamlit Frontend...")
# We disable CORS and XSRF protection for better compatibility with tunnels
!streamlit run frontend/app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false